In [134]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [135]:
import pandas as pd
import numpy as np

In [136]:
# read initial data sets
portfolios = pd.read_csv('/content/drive/MyDrive/25_Portfolios_5x5_cleaned.csv')
factors = pd.read_csv('/content/drive/MyDrive/F-F_Research_Data_Factors_cleaned.csv')
consumption = pd.read_csv('/content/drive/MyDrive/PCND.csv')
population = pd.read_csv('/content/drive/MyDrive/population.csv')

In [143]:
# DATA ENGINEERING - CONSUMPTION

consumption = consumption.rename(columns={'PCND': 'consumption_billions'})
population = population.rename(columns={'B230RC0Q173SBEA': 'population_thousands'})

# merge datasets

cons_pc = pd.merge(
    consumption,
    population,
    on='observation_date',
    how='inner'
)

# adjust consumption units

cons_pc['consumption_dollars'] = (
    cons_pc['consumption_billions'] * 1_000_000_000
)

# adjust population units

cons_pc['population_persons'] = (
    cons_pc['population_thousands'] * 1_000
)

# per capita consumption

cons_pc['consumption_per_capita'] = (
    cons_pc['consumption_dollars']
    / cons_pc['population_persons']
)

cons_pc['log_consumption_pc'] = np.log(
    cons_pc['consumption_per_capita']
)

cons_pc['cons_growth'] = (
    cons_pc['log_consumption_pc'].diff()
)

cons_pc['cons_growth_lagged'] = (
    cons_pc['cons_growth'].shift(1)
)


print(cons_pc.head())

  observation_date  consumption_billions  population_thousands  \
0       1947-01-01                74.872                143143   
1       1947-04-01                76.897                143790   
2       1947-07-01                78.649                144449   
3       1947-10-01                79.965                145122   
4       1948-01-01                81.546                145709   

   consumption_dollars  population_persons  consumption_per_capita  \
0         7.487200e+10           143143000              523.057362   
1         7.689700e+10           143790000              534.786842   
2         7.864900e+10           144449000              544.475905   
3         7.996500e+10           145122000              551.019143   
4         8.154600e+10           145709000              559.649713   

   log_consumption_pc  cons_growth  cons_growth_lagged  
0            6.259691          NaN                 NaN  
1            6.281868     0.022177                 NaN  
2          

In [142]:
# PORTFOLIO RETURNS DATA ENGINEERING

# rename index to 'Date'
portfolios = portfolios.rename(columns={'Unnamed: 0': 'Date'})
factors = factors.rename(columns={'Unnamed: 0': 'Date'})
portfolios['Date'] = pd.to_datetime(portfolios['Date'], format='%Y%m', errors='coerce')
factors['Date'] = pd.to_datetime(factors['Date'], format='%Y%m', errors='coerce')

# adjust returns units
portfolio_return_cols = portfolios.select_dtypes(include=np.number).columns

portfolios[portfolio_return_cols] = portfolios[portfolio_return_cols] / 100
factors['RF'] = factors['RF'] / 100

# Merge datasets
ff_data = pd.merge(portfolios, factors[['Date', 'RF']], on='Date', how='inner')


# cumulate returns to quarterly

ff_data['year'] = ff_data['Date'].dt.year
ff_data['quarter'] = ff_data['Date'].dt.quarter

def compound_returns(series):
    return (1 + series).prod() - 1

quarterly_portfolios = (
    ff_data
    .groupby(['year', 'quarter'])[portfolio_return_cols]
    .apply(compound_returns)
    .reset_index()
)

quarterly_rf = (
    ff_data
    .groupby(['year', 'quarter'])['RF']
    .apply(compound_returns)
    .reset_index()
)

quarterly_ff = pd.merge(
    quarterly_portfolios,
    quarterly_rf,
    on=['year', 'quarter'],
    how='inner'
)

for col in portfolio_return_cols:
    quarterly_ff[col] = quarterly_ff[col] - quarterly_ff['RF']


quarterly_ff['observation_date'] = pd.PeriodIndex(
    year=quarterly_ff['year'],
    quarter=quarterly_ff['quarter'],
    freq='Q'
).to_timestamp()


quarterly_ff_excess_returns = quarterly_ff.drop(
    columns=['year', 'quarter', 'RF']
)

quarterly_ff_excess_returns.head()

/tmp/ipykernel_1268/3475975601.py:52: FutureWarning: Constructing PeriodIndex from fields is deprecated. Use PeriodIndex.from_fields instead.
  quarterly_ff['observation_date'] = pd.PeriodIndex(


,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,SMALL HiBM,ME2 BM1,ME2 BM2,ME2 BM3,ME2 BM4,ME2 BM5,...,ME4 BM2,ME4 BM3,ME4 BM4,ME4 BM5,BIG LoBM,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM,observation_date
0,-0.000173,-0.001311,-0.000313,-0.000397,0.001063,0.000019,0.000043,0.000488,-0.000637,0.000420,...,0.000418,0.000084,0.000319,0.000798,0.000235,0.001326,0.000344,0.000717,0.000519,1926-07-01
1,0.001027,-0.000170,-0.000197,0.000146,-0.000209,0.000249,-0.000291,-0.000210,-0.000168,0.000571,...,0.000143,0.000248,0.000397,-0.000428,0.000214,0.000210,0.000035,0.000432,-0.000176,1926-10-01
2,0.000898,-0.000951,-0.000301,0.000358,0.000756,0.000124,0.000229,0.000525,0.000204,0.001718,...,0.000510,-0.000034,0.000483,0.002189,0.000246,0.000663,0.000206,0.000536,0.000583,1927-01-01
3,0.001369,0.002184,0.000311,0.001294,0.001133,0.000684,-0.000053,0.000890,0.000683,0.001083,...,0.000399,-0.000204,0.000415,0.001466,0.000752,0.000161,0.000167,0.000112,-0.000495,1927-04-01
4,0.003046,0.000488,0.000517,0.000092,0.000592,0.000866,0.001155,0.000488,0.000813,0.000740,...,0.001284,0.001222,0.000836,0.000420,0.002218,0.001032,0.001196,0.001268,0.002217,1927-07-01


In [139]:
# MERGE RETURNS AND CONSUMPTION DATASETS

cons_pc['observation_date'] = pd.to_datetime(cons_pc['observation_date'])

combined_dataset = pd.merge(
    quarterly_ff_excess_returns,
    cons_pc,
    on='observation_date',
    how='inner'
)

#rearrange columns to satisfy my ocd

consumption_cols = [
    'observation_date',
    'consumption_billions',
    'population_thousands',
    'consumption_dollars',
    'population_persons',
    'consumption_per_capita'
]

other_cols = [col for col in combined_dataset.columns if col not in consumption_cols]

new_column_order = consumption_cols + other_cols

combined_dataset = combined_dataset[new_column_order]

combined_dataset.head()

,observation_date,consumption_billions,population_thousands,consumption_dollars,population_persons,consumption_per_capita,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,...,ME4 BM3,ME4 BM4,ME4 BM5,BIG LoBM,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM,log_consumption_pc,cons_growth
0,1947-01-01,74.872,143143,7.487200e+10,143143000,523.057362,0.012737,-0.010558,-0.031992,0.010294,...,-0.032071,0.006209,-0.007486,0.011958,-0.027717,-0.024066,-0.016110,-0.033898,6.259691,NaN
1,1947-04-01,76.897,143790,7.689700e+10,143790000,534.786842,-0.110057,-0.134536,-0.099850,-0.115021,...,-0.039192,-0.082707,-0.050863,0.001832,0.015858,-0.029538,0.050063,-0.019045,6.281868,0.022177
2,1947-07-01,78.649,144449,7.864900e+10,144449000,544.475905,0.085437,0.044049,0.033609,0.034605,...,0.070158,0.049779,0.086857,0.008584,0.009521,-0.003738,0.020855,0.033434,6.299824,0.017955
3,1947-10-01,79.965,145122,7.996500e+10,145122000,551.019143,-0.060114,0.020601,0.018010,0.019557,...,0.031496,0.000852,-0.008834,0.014859,0.060376,0.045773,0.046850,0.095099,6.311770,0.011946
4,1948-01-01,81.546,145709,8.154600e+10,145709000,559.649713,0.067091,-0.003656,-0.012411,0.009131,...,0.014221,0.038338,0.057488,-0.028538,0.005230,-0.014079,0.006078,0.016352,6.327311,0.015542


In [140]:
# Filter data into paper and extended datasets (paper = up until 1999, extended is after that)

ff_paper = combined_dataset[
    (combined_dataset['observation_date'] >= '1947-04-01' ) &
    (combined_dataset['observation_date'] <= '1999-10-01')
].copy()

ff_extended = combined_dataset[
    (combined_dataset['observation_date'] > '1999-10-01' )
].copy()

ff_paper.head()

,observation_date,consumption_billions,population_thousands,consumption_dollars,population_persons,consumption_per_capita,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,...,ME4 BM3,ME4 BM4,ME4 BM5,BIG LoBM,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM,log_consumption_pc,cons_growth
1,1947-04-01,76.897,143790,7.689700e+10,143790000,534.786842,-0.110057,-0.134536,-0.099850,-0.115021,...,-0.039192,-0.082707,-0.050863,0.001832,0.015858,-0.029538,0.050063,-0.019045,6.281868,0.022177
2,1947-07-01,78.649,144449,7.864900e+10,144449000,544.475905,0.085437,0.044049,0.033609,0.034605,...,0.070158,0.049779,0.086857,0.008584,0.009521,-0.003738,0.020855,0.033434,6.299824,0.017955
3,1947-10-01,79.965,145122,7.996500e+10,145122000,551.019143,-0.060114,0.020601,0.018010,0.019557,...,0.031496,0.000852,-0.008834,0.014859,0.060376,0.045773,0.046850,0.095099,6.311770,0.011946
4,1948-01-01,81.546,145709,8.154600e+10,145709000,559.649713,0.067091,-0.003656,-0.012411,0.009131,...,0.014221,0.038338,0.057488,-0.028538,0.005230,-0.014079,0.006078,0.016352,6.327311,0.015542
5,1948-04-01,83.169,146289,8.316900e+10,146289000,568.525316,0.072603,0.197612,0.124719,0.109120,...,0.134883,0.050218,0.155405,0.101079,0.114926,0.115030,0.099414,0.192004,6.343046,0.015735


In [141]:
ff_extended.head()

,observation_date,consumption_billions,population_thousands,consumption_dollars,population_persons,consumption_per_capita,SMALL LoBM,ME1 BM2,ME1 BM3,ME1 BM4,...,ME4 BM3,ME4 BM4,ME4 BM5,BIG LoBM,ME5 BM2,ME5 BM3,ME5 BM4,BIG HiBM,log_consumption_pc,cons_growth
212,2000-01-01,1492.211,281304,1.492211e+12,281304000,5304.620624,0.277401,0.218752,0.146090,0.194657,...,0.059643,0.075719,-0.020084,0.030296,-0.004114,-0.052516,-0.022939,0.007626,8.576334,0.003867
213,2000-04-01,1535.118,282002,1.535118e+12,282002000,5443.642244,-0.165593,-0.059047,-0.018635,-0.032509,...,-0.030009,-0.049772,-0.033197,-0.037771,-0.099620,-0.025230,-0.059364,-0.039824,8.602204,0.025870
214,2000-07-01,1557.492,282769,1.557492e+12,282769000,5508.001231,-0.123777,0.047488,-0.001578,0.049741,...,0.077046,0.099085,0.064721,-0.036777,0.030947,0.133905,0.221888,0.048737,8.613957,0.011753
215,2000-10-01,1577.579,283518,1.577579e+12,283518000,5564.299268,-0.383137,-0.224461,-0.105033,-0.083076,...,0.134888,0.107499,0.152421,-0.152299,0.025701,-0.080329,0.078922,0.027984,8.624126,0.010169
216,2001-01-01,1572.434,284169,1.572434e+12,284169000,5533.446646,-0.048781,0.031498,0.070540,0.114179,...,-0.031553,-0.058526,0.074779,-0.180859,-0.033843,0.032803,-0.061568,0.010550,8.618566,-0.005560
